In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import xgboost as xgb
import lightgbm as lgb
from sklearn.svm import SVC

In [3]:
with open("original_RT_IoT2022.csv") as f:
    print(repr(f.readline()))

'",id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,bwd_pkts_per_sec,flow_pkts_per_sec,down_up_ratio,fwd_header_size_tot,fwd_header_size_min,fwd_header_size_max,bwd_header_size_tot,bwd_header_size_min,bwd_header_size_max,flow_FIN_flag_count,flow_SYN_flag_count,flow_RST_flag_count,fwd_PSH_flag_count,bwd_PSH_flag_count,flow_ACK_flag_count,fwd_URG_flag_count,bwd_URG_flag_count,flow_CWR_flag_count,flow_ECE_flag_count,fwd_pkts_payload.min,fwd_pkts_payload.max,fwd_pkts_payload.tot,fwd_pkts_payload.avg,fwd_pkts_payload.std,bwd_pkts_payload.min,bwd_pkts_payload.max,bwd_pkts_payload.tot,bwd_pkts_payload.avg,bwd_pkts_payload.std,flow_pkts_payload.min,flow_pkts_payload.max,flow_pkts_payload.tot,flow_pkts_payload.avg,flow_pkts_payload.std,fwd_iat.min,fwd_iat.max,fwd_iat.tot,fwd_iat.avg,fwd_iat.std,bwd_iat.min,bwd_iat.max,bwd_iat.tot,bwd_iat.avg,bwd_iat.std,flow_iat.min,flow_iat.max,flow_iat.tot,flow_iat.avg,flow_iat.std,

In [4]:
import pandas as pd

df = pd.read_csv("original_RT_IoT2022.csv")

print(repr(df.columns.tolist()))

[',id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,bwd_pkts_per_sec,flow_pkts_per_sec,down_up_ratio,fwd_header_size_tot,fwd_header_size_min,fwd_header_size_max,bwd_header_size_tot,bwd_header_size_min,bwd_header_size_max,flow_FIN_flag_count,flow_SYN_flag_count,flow_RST_flag_count,fwd_PSH_flag_count,bwd_PSH_flag_count,flow_ACK_flag_count,fwd_URG_flag_count,bwd_URG_flag_count,flow_CWR_flag_count,flow_ECE_flag_count,fwd_pkts_payload.min,fwd_pkts_payload.max,fwd_pkts_payload.tot,fwd_pkts_payload.avg,fwd_pkts_payload.std,bwd_pkts_payload.min,bwd_pkts_payload.max,bwd_pkts_payload.tot,bwd_pkts_payload.avg,bwd_pkts_payload.std,flow_pkts_payload.min,flow_pkts_payload.max,flow_pkts_payload.tot,flow_pkts_payload.avg,flow_pkts_payload.std,fwd_iat.min,fwd_iat.max,fwd_iat.tot,fwd_iat.avg,fwd_iat.std,bwd_iat.min,bwd_iat.max,bwd_iat.tot,bwd_iat.avg,bwd_iat.std,flow_iat.min,flow_iat.max,flow_iat.tot,flow_iat.avg,flow_iat.std,

In [5]:
import io

with open("original_RT_IoT2022.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()

cleaned_lines = []
for line in lines:
    line = line.rstrip("\n").rstrip("\r")
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    cleaned_lines.append(line)

df = pd.read_csv(io.StringIO("\n".join(cleaned_lines)))
print(df.shape)
print(df.columns.tolist())

(123117, 85)
['Unnamed: 0', 'id.orig_p', 'id.resp_p', 'proto', 'service', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'down_up_ratio', 'fwd_header_size_tot', 'fwd_header_size_min', 'fwd_header_size_max', 'bwd_header_size_tot', 'bwd_header_size_min', 'bwd_header_size_max', 'flow_FIN_flag_count', 'flow_SYN_flag_count', 'flow_RST_flag_count', 'fwd_PSH_flag_count', 'bwd_PSH_flag_count', 'flow_ACK_flag_count', 'fwd_URG_flag_count', 'bwd_URG_flag_count', 'flow_CWR_flag_count', 'flow_ECE_flag_count', 'fwd_pkts_payload.min', 'fwd_pkts_payload.max', 'fwd_pkts_payload.tot', 'fwd_pkts_payload.avg', 'fwd_pkts_payload.std', 'bwd_pkts_payload.min', 'bwd_pkts_payload.max', 'bwd_pkts_payload.tot', 'bwd_pkts_payload.avg', 'bwd_pkts_payload.std', 'flow_pkts_payload.min', 'flow_pkts_payload.max', 'flow_pkts_payload.tot', 'flow_pkts_payload.avg', 'flow_pkts_payload.std', 'fwd_iat.min', 'fwd_iat.max'

In [6]:
cols_to_drop = ["Unnamed: 0", "Flow_ID", "Source_IP", "Destination_IP",
                "Timestamp", "id.orig_p", "id.resp_p"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


In [7]:
feat_cols_for_dedup = [c for c in df.columns if c != "Attack_type"]
before = len(df)
df = df.drop_duplicates(subset=feat_cols_for_dedup).reset_index(drop=True)
print(f"Rows before dedup: {before}  |  Rows after dedup: {len(df)}")

Rows before dedup: 123117  |  Rows after dedup: 18272


In [8]:
normal_traffic = ["Thing_Speak", "Wipro_bulb", "MQTT_Publish"]
df["Attack_type"] = df["Attack_type"].apply(lambda x: 0 if x in normal_traffic else 1)
print("\nClass distribution (FULL deduplicated data):")
print(df["Attack_type"].value_counts(normalize=True) * 100)


Class distribution (FULL deduplicated data):
Attack_type
0    65.449869
1    34.550131
Name: proportion, dtype: float64


In [9]:
label_encoders = {}
for col in df.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

X = df.drop(columns=["Attack_type"])
y = df["Attack_type"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {len(X_train)}  |  Test size: {len(X_test)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Train size: 14617  |  Test size: 3655


In [11]:
results = {}

print("\n--- Random Forest ---")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced")
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred, target_names=["Normal","Attack"]))
cv = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f"5-Fold CV Accuracy: {cv.mean():.4f} (+/- {cv.std()*2:.4f})")
results['RF'] = dict(pred=rf_pred, probs=rf.predict_proba(X_test_scaled)[:,1], cv=cv)

print("\n--- XGBoost ---")
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred, target_names=["Normal","Attack"]))
cv = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f"5-Fold CV Accuracy: {cv.mean():.4f} (+/- {cv.std()*2:.4f})")
results['XGB'] = dict(pred=xgb_pred, probs=xgb_model.predict_proba(X_test)[:,1], cv=cv)

print("\n--- LightGBM ---")
lgb_model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, lgb_pred))
print(classification_report(y_test, lgb_pred, target_names=["Normal","Attack"]))
cv = cross_val_score(lgb_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f"5-Fold CV Accuracy: {cv.mean():.4f} (+/- {cv.std()*2:.4f})")
results['LGB'] = dict(pred=lgb_pred, probs=lgb_model.predict_proba(X_test)[:,1], cv=cv)

print("\n--- SVM (on balanced training subsample for tractability) ---")
train_df_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
train_df_scaled['Target'] = y_train.values
normal_train = train_df_scaled[train_df_scaled['Target'] == 0]
attack_train = train_df_scaled[train_df_scaled['Target'] == 1]
n_samples = min(len(normal_train), len(attack_train), 3000)
svm_train_data = pd.concat([
    normal_train.sample(n_samples, random_state=42),
    attack_train.sample(n_samples, random_state=42)
]).sample(frac=1, random_state=42)
X_train_svm = svm_train_data.drop(columns=['Target'])
y_train_svm = svm_train_data['Target']
svm = SVC(kernel="rbf", C=1.0, probability=True, random_state=42)
svm.fit(X_train_svm, y_train_svm)
svm_pred = svm.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred, target_names=["Normal","Attack"]))
results['SVM'] = dict(pred=svm_pred, probs=svm.predict_proba(X_test_scaled)[:,1])



--- Random Forest ---
Test Accuracy: 0.9942544459644322
              precision    recall  f1-score   support

      Normal       1.00      0.99      1.00      2392
      Attack       0.99      0.99      0.99      1263

    accuracy                           0.99      3655
   macro avg       0.99      0.99      0.99      3655
weighted avg       0.99      0.99      0.99      3655

5-Fold CV Accuracy: 0.9915 (+/- 0.0021)

--- XGBoost ---
Test Accuracy: 0.9909712722298222
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99      2392
      Attack       0.98      0.99      0.99      1263

    accuracy                           0.99      3655
   macro avg       0.99      0.99      0.99      3655
weighted avg       0.99      0.99      0.99      3655

5-Fold CV Accuracy: 0.9904 (+/- 0.0057)

--- LightGBM ---
[LightGBM] [Info] Number of positive: 5050, number of negative: 9567
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead 

C:\Users\ADITI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2750: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


Test Accuracy: 0.9578659370725034
              precision    recall  f1-score   support

      Normal       0.98      0.96      0.97      2392
      Attack       0.92      0.96      0.94      1263

    accuracy                           0.96      3655
   macro avg       0.95      0.96      0.95      3655
weighted avg       0.96      0.96      0.96      3655



C:\Users\ADITI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2750: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


In [12]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

print("="*60)
print("Support Vector Machine (Baseline)")
print("="*60)

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, svm_pred))

print("\nClassification Report:")
print(classification_report(y_test, svm_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

cv_scores = cross_val_score(
    svm_model,
    X_train_scaled,
    y_train,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

print("\nCross Validation Accuracy:")
print(cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

Support Vector Machine (Baseline)
Accuracy: 0.9704514363885088

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      2392
           1       0.94      0.97      0.96      1263

    accuracy                           0.97      3655
   macro avg       0.96      0.97      0.97      3655
weighted avg       0.97      0.97      0.97      3655


Confusion Matrix:
[[2318   74]
 [  34 1229]]

Cross Validation Accuracy:
[0.9630643  0.97161423 0.97194663 0.96955183 0.97092029]
Mean CV Accuracy: 0.9694194540867211


In [13]:
import pandas as pd

print("="*70)
print("DETECTION ENGINE COMPARISON -- Accuracy")
print("="*70)
summary_rows = []
for name, r in results.items():
    acc = accuracy_score(y_test, r['pred'])
    summary_rows.append({'Model': name, 'Accuracy': round(acc, 4)})
acc_df = pd.DataFrame(summary_rows)
print(acc_df.to_string(index=False))

DETECTION ENGINE COMPARISON -- Accuracy
Model  Accuracy
   RF    0.9943
  XGB    0.9910
  LGB    0.9907
  SVM    0.9579


In [14]:
from sklearn.model_selection import train_test_split as tts
import numpy as np

lgb_accs = []
for seed in [1, 2, 3, 4, 5]:
    Xtr, Xte, ytr, yte = tts(X, y, test_size=0.2, random_state=seed, stratify=y)
    lgb_seed = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, verbose=-1)
    lgb_seed.fit(Xtr, ytr)
    acc = accuracy_score(yte, lgb_seed.predict(Xte))
    lgb_accs.append(acc)
    print(f"Seed {seed}: LightGBM Accuracy={acc:.4f}")

print(f"\nLightGBM stability: mean={np.mean(lgb_accs):.4f}  std={np.std(lgb_accs):.4f}")

Seed 1: LightGBM Accuracy=0.9891
Seed 2: LightGBM Accuracy=0.9921
Seed 3: LightGBM Accuracy=0.9918
Seed 4: LightGBM Accuracy=0.9940
Seed 5: LightGBM Accuracy=0.9902

LightGBM stability: mean=0.9914  std=0.0017


In [15]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, 20, None],       # None = unlimited, like your current setup
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
    param_distributions=rf_param_dist,
    n_iter=25, cv=3, scoring='accuracy', n_jobs=-1, random_state=42
)
rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)
best_rf = rf_search.best_estimator_
rf_pred = best_rf.predict(X_test)
print("Tuned RF Test Accuracy:", accuracy_score(y_test, rf_pred))

Best RF params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}
Tuned RF Test Accuracy: 0.9920656634746922


In [16]:
import xgboost as xgb

xgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb.XGBClassifier(eval_metric='logloss'),
    param_distributions=xgb_param_dist,
    n_iter=25, cv=3, scoring='accuracy', n_jobs=-1, random_state=42
)
xgb_search.fit(X_train, y_train)

print("Best XGBoost params:", xgb_search.best_params_)
best_xgb = xgb_search.best_estimator_
xgb_pred = best_xgb.predict(X_test)
print("Tuned XGBoost Test Accuracy:", accuracy_score(y_test, xgb_pred))

Best XGBoost params: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
Tuned XGBoost Test Accuracy: 0.9934336525307798


In [17]:
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb

lgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 7, 9, -1],          # -1 = no limit
    'num_leaves': [15, 31, 63, 127],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'min_child_samples': [5, 10, 20]
}

lgb_search = RandomizedSearchCV(
    lgb.LGBMClassifier(verbose=-1),
    param_distributions=lgb_param_dist,
    n_iter=25, cv=3, scoring='accuracy', n_jobs=-1, random_state=42
)
lgb_search.fit(X_train, y_train)

print("Best LightGBM params:", lgb_search.best_params_)
best_lgb = lgb_search.best_estimator_
lgb_pred = best_lgb.predict(X_test)
print("Tuned LightGBM Test Accuracy:", accuracy_score(y_test, lgb_pred))

Best LightGBM params: {'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': -1, 'learning_rate': 0.2}
Tuned LightGBM Test Accuracy: 0.9958960328317373


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC

svm_param_dist = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}

svm_search = RandomizedSearchCV(
    SVC(probability=True, random_state=42),
    param_distributions=svm_param_dist,
    n_iter=8,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

svm_search.fit(X_train_scaled, y_train)

print("Best SVM Parameters:")
print(svm_search.best_params_)

best_svm = svm_search.best_estimator_

svm_pred = best_svm.predict(X_test_scaled)

print("Tuned SVM Accuracy:", accuracy_score(y_test, svm_pred))